In [1]:
!pip install discord
!pip install pyPDF2

In [2]:
import discord
import requests
import json
import PyPDF2
from keys import DiscordBotToken,DiscordWebhookUrl,OpenRouterToken,TOKEN,ChannelID,MODEL


code to read pdf 


In [3]:
from PyPDF2 import PdfReader

def readPdf(filepath):
    reader = PdfReader(filepath) 
    #print(len(reader.pages))

    text = ''

    # Read text from each page
    for page in reader.pages:
        text = text + page.extract_text()

    #print(text)

    return text




using openrouter for llm requests


In [4]:
def Summarizer(text):
    prompt = f" summarize this in under 700 words {text} " 

    headers = {

    "Authorization":f'Bearer {OpenRouterToken}',
    "Content-type": "application/json"

    }
    json ={
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}]
    }

    response = requests.post("https://openrouter.ai/api/v1/chat/completions",headers=headers,json=json)
    if response.status_code == 200:
        print("ai success")

        print(f""" {response.json()["choices"][0]["message"]["content"]}
                """)
        return response.json()["choices"][0]["message"]["content"]
    else:
        return f"Error: {response.status_code}"

using ollama for llm reqeuest

In [5]:
    
def Summarizerv2(text):
    
    url = "http://localhost:11434/api/chat"

    prompt = f"summarize the text in 300 characters {text}"

    payload = {

        "model": "gemma3:4b",
        "messages": [{"role":"user","content":prompt}],
        "stream" : False
    }

    response = requests.post(url,json=payload,stream=False)

    data = response.json()
    
    try:

        if response.status_code == 200:
            if "message" in data and "content" in data["message"]:
                output = data["message"]["content"]
                print(output)
                return output
    
    except json.JSONDecodeError:
        print("shit failed")





code to post to discord


In [6]:
def post_to_discord(text):

    
    response = requests.post(DiscordWebhookUrl,json={"content": text})

    if response.status_code == 204:
        print(" discord success")
    else :
        print("discord fail")

code for discord bot sensing + webhook posting . the @client.event etc etc are in discord docs . the asyncio,nest_asyncio etc etc are there to make webhooks work in ipynb 


In [7]:
import discord
import os
import nest_asyncio
import asyncio

# Patch the event loop to work inside Jupyter
nest_asyncio.apply()

# Set up your bot with appropriate intents
intents = discord.Intents.default()
intents.message_content = True
client = discord.Client(intents=intents)

@client.event
async def on_ready():
    print(f'✅ Logged in as {client.user}')

@client.event
async def on_message(message):
    # Avoid responding to self
    if message.author == client.user:
        return

    # Check if there's an attachment and it's a PDF
    for attachment in message.attachments:
        print(f"📄 PDF received: {attachment.filename}")

        #  Download the PDF to your local PC
        file_path = os.path.join("downloads", attachment.filename)
        os.makedirs("downloads", exist_ok=True)
        await attachment.save(file_path)
        
        text = readPdf(filepath=file_path)
        #text1 = Summarizer(text)
        text1 = Summarizerv2(text)
        
        post_to_discord(text1)
        

        print(f"💾 Saved to {file_path}")
        break

        
            
            


TOKEN = DiscordBotToken


loop = asyncio.get_event_loop()
loop.create_task(client.start(TOKEN))


<Task pending name='Task-1' coro=<Client.start() running at c:\Users\adith\AppData\Local\Programs\Python\Python311\Lib\site-packages\discord\client.py:825>>

✅ Logged in as zacrobot#4121


code for testing if discord part works

In [8]:

dummytext = """  TEXT TO TEST DISCORD CHARACTER LIMIT
"""
response = requests.post(DiscordWebhookUrl,json={"content": dummytext})

if response.status_code == 204:
    print(" discord success")
else :
    print("discord fail")

 discord success
